In [ ]:
! pip install datasets

In [25]:
import torch
from torch.optim import AdamW
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from transformers import BertModel, BertTokenizer, BertPreTrainedModel
from datasets import load_dataset, ClassLabel
from tqdm import tqdm

In [ ]:
def prepare_dataset():
    dataset = load_dataset("Deeppavlov/clinc150")

    print("Sample example:", dataset["train"][0])

    print(f"Train size before filtering: {len(dataset['train'])}")
    print(f"Validation size before filtering: {len(dataset['validation'])}")
    print(f"Test size before filtering: {len(dataset['test'])}")

    # As I saw some label were of NoneType, so I removed them
    def filter_none_labels(example):
        return example["label"] is not None

    dataset = dataset.filter(
        filter_none_labels,
        batched=False
    )

    print(f"Train size after filtering: {len(dataset['train'])}")
    print(f"Validation size after filtering: {len(dataset['validation'])}")
    print(f"Test size after filtering: {len(dataset['test'])}")

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    def tokenize_fn(examples):
        return tokenizer(
            examples["utterance"],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

    tokenized_datasets = dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=["utterance"]
    )

    tokenized_datasets.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "label"])

    return (
        tokenized_datasets["train"],
        tokenized_datasets["validation"],
        tokenized_datasets["test"]
    )

train_data, validation_data, test_data = prepare_dataset()

def create_dataloader(dataset, batch_size = 32):
    return DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = True,
        num_workers = 2
    )

In [ ]:
class Config:
    # Model configurations
    bottleneck_dims = [8, 32, 64, 128]
    bert_model = "bert-base-uncased"
    num_labels = 150
    dropout_prob = 0.1

    # Training configurations
    batch_size = 32
    bert_lr = 2e-5
    bottleneck_lr = 1e-3
    epochs = 5
    max_seq_length = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# *Part1*

In [ ]:
class BertBottleneckClassifier(nn.Module):
    def __init__(self, bottleneck_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained(Config.bert_model)
        self.bottleneck = nn.Linear(self.bert.config.hidden_size, bottleneck_dim)
        # self.dropout = nn.Dropout(Config.dropout_prob)
        self.classifier = nn.Linear(bottleneck_dim, Config.num_labels)

        # Freeze BERT parameters initially
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        bert_out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        cls_embedding = bert_out.last_hidden_state[:, 0, :]
        compressed = self.bottleneck(cls_embedding)
        # compressed = self.dropout(compressed)
        return self.classifier(compressed)


def train_model(bottleneck_dim):
    model = BertBottleneckClassifier(bottleneck_dim).to(device)

    optimizer = AdamW([
        {"params": model.bert.parameters(), "lr": Config.bert_lr},
        {"params": model.bottleneck.parameters(), "lr": Config.bottleneck_lr},
        {"params": model.classifier.parameters(), "lr": Config.bottleneck_lr}
    ])

    loss_fn = nn.CrossEntropyLoss()

    train_loader = create_dataloader(train_data, Config.batch_size)
    val_loader = create_dataloader(validation_data, Config.batch_size)


    best_val_acc = 0
    best_model = None

    for epoch in range(Config.epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc = f"Training X = {bottleneck_dim} ----> Epoch = {epoch + 1}"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(**inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Model evaluation on validation dataset
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
                labels = batch["label"].to(device)

                outputs = model(**inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total
        print(f"Epoch = {epoch+1}: Val Acc = {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model.state_dict()

    # Load best model
    model.load_state_dict(best_model)
    return model

def test_model(model):
    test_loader = create_dataloader(test_data, Config.batch_size)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(device)

            outputs = model(**inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total


results = {}

for dim in Config.bottleneck_dims:
    print(f"\n{'-'*50}")
    print(f"Bottleneck dimension: {dim}")
    print(f"{'-'*50}")

    model = train_model(dim)
    test_acc = test_model(model)
    results[dim] = test_acc
    print(f"Test Accuracy for X={dim}: {test_acc:.4f}")


dims = sorted(results.keys())
part1_accs = [results[d] for d in dims]

plt.figure(figsize=(10, 6))
plt.plot(dims, part1_accs, 'bo-')
plt.xscale('log', base=2)
plt.xticks(dims, labels=dims)
plt.xlabel('Bottleneck Dimension (X)')
plt.ylabel('Test Accuracy')
plt.title('Classification Accuracy vs Bottleneck Width (Bottleneck without Reconstruction)')
plt.grid(True)
info_text = f"Epochs: {Config.epochs}\nBatch Size: {Config.batch_size}\nTraining Pairs: {len(train_data)}\nTest Pairs: {len(test_data)}\nValidation Pairs: {len(validation_data)}"
plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.show()

# Part 2

In [ ]:
class BertBottleneckAutoencoderClassifier(nn.Module):
    def __init__(self, bottleneck_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained(Config.bert_model)

        self.encoder = nn.Sequential(
            nn.Linear(768, 384),
            nn.ReLU(),
            nn.Linear(384, bottleneck_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 384),
            nn.ReLU(),
            nn.Linear(384, 768)
        )

        self.classifier = nn.Sequential(
            # nn.Dropout(Config.dropout_prob),
            nn.Linear(bottleneck_dim, Config.num_labels)
        )

        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        bert_out = self.bert(input_ids, attention_mask, token_type_ids)
        cls_embedding = bert_out.last_hidden_state[:, 0, :]

        compressed = self.encoder(cls_embedding)
        reconstructed = self.decoder(compressed)

        return {
            'logits': self.classifier(compressed),
            'compressed': compressed,
            'reconstructed': reconstructed,
            'original': cls_embedding
        }

def train_autoencoder(bottleneck_dim):
    model = BertBottleneckAutoencoderClassifier(bottleneck_dim).to(device)

    optimizer = AdamW([
        {'params': model.bert.parameters(), 'lr': Config.bert_lr},
        {'params': model.encoder.parameters(), 'lr': Config.bottleneck_lr},
        {'params': model.decoder.parameters(), 'lr': Config.bottleneck_lr},
        {'params': model.classifier.parameters(), 'lr': Config.bottleneck_lr}
    ])

    train_loader = create_dataloader(train_data, Config.batch_size)
    val_loader = create_dataloader(validation_data, Config.batch_size)

    best_val_acc = 0
    best_model = None

    for epoch in range(Config.epochs):
        model.train()
        for batch in tqdm(train_loader, desc = f"Training X = {bottleneck_dim} ----> Epoch = {epoch + 1}"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch['label'].to(device)

            outputs = model(**inputs)
            loss_cls = nn.CrossEntropyLoss()(outputs['logits'], labels)
            loss_recon = nn.MSELoss()(outputs['reconstructed'], outputs['original'])
            total_loss = loss_cls + loss_recon

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

        # Model evaluation on validation dataset
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader:
              inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
              labels = batch['label'].to(device)

              outputs = model(**inputs)
              _, predicted = torch.max(outputs['logits'], 1)
              total += labels.size(0)
              correct += (predicted == labels).sum().item()

        val_acc = correct / total
        print(f"Epoch = {epoch+1}: Val Acc = {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model.state_dict()

    # Load best model
    model.load_state_dict(best_model)
    return model

def test_model(model):
    test_loader = create_dataloader(test_data, Config.batch_size)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(device)

            outputs = model(**inputs)
            _, predicted = torch.max(outputs['logits'], 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

results = {}

for dim in Config.bottleneck_dims:
    print(f"\n{'-'*50}")
    print(f"Bottleneck dimension: {dim}")
    print(f"{'-'*50}")

    model = train_autoencoder(dim)
    test_acc = test_model(model)
    results[dim] = test_acc
    print(f'Test Accuracy for X={dim}: {test_acc:.4f}')

dims = sorted(results.keys())
part2_accs = [results[d] for d in dims]

plt.figure(figsize=(10, 6))
plt.plot(dims, part2_accs, 'bo-')
plt.xscale('log', base = 2)
plt.xticks(dims, labels = dims)
plt.xlabel('Bottleneck Dimension (X)')
plt.ylabel('Test Accuracy')
plt.title('Classification Accuracy vs Bottleneck Width (Bottleneck with Reconstruction)')
plt.legend()
plt.grid(True)
info_text = f"Epochs: {Config.epochs}\nBatch Size: {Config.batch_size}\nTraining Pairs: {len(train_data)}\nValidation Pairs: {len(test_data)}\nTest Pairs: {len(validation_data)}"
plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(dims, part1_accs, 'ro-', label='Part 1 Accuracy')
plt.plot(dims, part2_accs, 'bo-', label='Part 2 Accuracy')

plt.xscale('log', base=2)
plt.xticks(dims, labels=dims)
plt.xlabel('Bottleneck Dimension (X)')
plt.ylabel('Test Accuracy')
plt.title('Classification Accuracy vs Bottleneck Width')
plt.legend()
plt.grid(True)
info_text = f"Epochs: {Config.epochs}\nBatch Size: {Config.batch_size}\nTraining Pairs: {len(train_data)}\nTest Pairs: {len(test_data)}\nValidation Pairs: {len(validation_data)}"
plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.show()